In [1]:
from __future__ import division, print_function
import sys, os, glob, time, warnings, gc
import numpy as np
# import matplotlib
# matplotlib.use("Agg")
import matplotlib.pyplot as plt
from astropy.table import Table, vstack, hstack, join
import fitsio
# from astropy.io import fits

In [2]:
params = {'legend.fontsize': 'large',
         'axes.labelsize': 'large',
         'axes.titlesize':'large',
         'xtick.labelsize':'large',
         'ytick.labelsize':'large',
         'figure.facecolor':'w'} 
plt.rcParams.update(params)

In [14]:
cat = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_targets/lrg_paper/desi_lrg_catalog_for_calculating_zp_sensitivity.fits'))
lrgmask = Table(fitsio.read('/Users/rongpu/Documents/Data/desi_targets/lrg_paper/desi_lrg_catalog_for_calculating_zp_sensitivity_lrgmask_v1.fits'))
cat = hstack([cat, lrgmask], join_type='exact')
print(len(cat))

13029189


In [15]:
mask = cat['lrg_mask']==0
print(np.sum(~mask)/len(mask))
cat = cat[mask]
print(len(cat))

0.12225104724476711
11436357


In [16]:
cat['gmag'] = 22.5 - 2.5 * np.log10((cat['FLUX_G'] / cat['MW_TRANSMISSION_G']).clip(1e-7))
cat['rmag'] = 22.5 - 2.5 * np.log10((cat['FLUX_R'] / cat['MW_TRANSMISSION_R']).clip(1e-7))
cat['zmag'] = 22.5 - 2.5 * np.log10((cat['FLUX_Z'] / cat['MW_TRANSMISSION_Z']).clip(1e-7))
cat['w1mag'] = 22.5 - 2.5 * np.log10((cat['FLUX_W1'] / cat['MW_TRANSMISSION_W1']).clip(1e-7))
cat['zfibermag'] = 22.5 - 2.5 * np.log10((cat['FIBERFLUX_Z'] / cat['MW_TRANSMISSION_Z']).clip(1e-7))

In [17]:
def get_lrgs(cat):

    mask_quality = np.full(len(cat), True)

    mask_quality &= (cat['FLUX_IVAR_R'] > 0) & (cat['FLUX_R'] > 0)   # ADM quality in r.
    mask_quality &= (cat['FLUX_IVAR_Z'] > 0) & (cat['FLUX_Z'] > 0) & (cat['FIBERFLUX_Z'] > 0)   # ADM quality in z.
    mask_quality &= (cat['FLUX_IVAR_W1'] > 0) & (cat['FLUX_W1'] > 0)  # ADM quality in W1.

    mask_quality &= (cat['GAIA_PHOT_G_MEAN_MAG'] == 0) | (cat['GAIA_PHOT_G_MEAN_MAG'] > 18)  # remove bright GAIA sources

    # ADM remove stars with zfibertot < 17.5 that are missing from GAIA.
    mask_quality &= cat['FIBERTOTFLUX_Z'] < 10**(-0.4*(17.5-22.5))

    # ADM observed in every band.
    mask_quality &= (cat['NOBS_G'] > 0) & (cat['NOBS_R'] > 0) & (cat['NOBS_Z'] > 0)

    # Apply masks
    maskbits = [1, 12, 13]
    mask_clean = np.ones(len(cat), dtype=bool)
    for bit in maskbits:
        mask_clean &= (cat['MASKBITS'] & 2**bit)==0
    # print(np.sum(~mask_clean)/len(mask_clean))
    mask_quality &= mask_clean

    ######################### Selection cuts #########################
    gmag = cat['gmag']
    rmag = cat['rmag']
    zmag = cat['zmag']
    w1mag = cat['w1mag']
    zfibermag = cat['zfibermag']

    mask_south = cat['PHOTSYS']=='S'
    mask_south &= zmag - w1mag > 0.8 * (rmag - zmag) - 0.6  # non-stellar cut
    mask_south &= zfibermag < 21.6                   # faint limit
    mask_south &= (gmag - w1mag > 2.9) | (rmag - w1mag > 1.8)  # low-z cuts
    mask_south &= (
        ((rmag - w1mag > (w1mag - 17.14) * 1.8)
         & (rmag - w1mag > (w1mag - 16.33) * 1.))
        | (rmag - w1mag > 3.3)
    )  # double sliding cuts and high-z extension
    mask_north = cat['PHOTSYS']=='N'
    mask_north &= zmag - w1mag > 0.8 * (rmag - zmag) - 0.6  # non-stellar cut
    mask_north &= zfibermag < 21.61                   # faint limit
    mask_north &= (gmag - w1mag > 2.97) | (rmag - w1mag > 1.8)  # low-z cuts
    mask_north &= (
        ((rmag - w1mag > (w1mag - 17.13) * 1.83)
         & (rmag - w1mag > (w1mag - 16.31) * 1.))
        | (rmag - w1mag > 3.4)
    )  # double sliding cuts and high-z extension

    mask_lrg = mask_south | mask_north
    
    #################################################################

    mask_lrg &= mask_quality

    return mask_lrg

In [18]:
count0 = np.sum(get_lrgs(cat))
print(count0, count0/len(cat))

10817704 0.9459047142372348


In [36]:
dn_dict = {}

for band in ['gmag', 'rmag', 'zmag', 'w1mag']:

    cat1 = cat.copy()
    cat1[band] = cat1[band] - 0.005
    if band=='zmag':
        cat1['zfibermag'] = cat1['zfibermag'] - 0.005
    count1 = np.sum(get_lrgs(cat1))
    
    cat1 = cat.copy()
    cat1[band] = cat1[band] + 0.005
    if band=='zmag':
        cat1['zfibermag'] = cat1['zfibermag'] + 0.005
    count2 = np.sum(get_lrgs(cat1))
    
    fractional_diff = (count2-count1)/count0
    dn_dict[band.replace('mag', '')] = fractional_diff
    
    sign = '+' if fractional_diff>=0 else '-'
    
    print('Net change of +0.01 in {}: {}{:.2f}% change in target density'.format(band, sign, np.abs(fractional_diff)*100))

Net change of +0.01 in gmag: +0.11% change in target density
Net change of +0.01 in rmag: +1.40% change in target density
Net change of +0.01 in zmag: -1.23% change in target density
Net change of +0.01 in w1mag: -2.89% change in target density


In [37]:
dn_dict

{'g': 0.001120200737605688,
 'r': 0.013979306514580174,
 'z': -0.012331729542609042,
 'w1': -0.02893543768622251}

In [38]:
# zero point uncertainties in mmag
zp_err_dict = {'g':3, 'r':3, 'z':6, 'w1':1}

In [43]:
total_rms_squared = 0.
for band in ['g', 'r', 'z', 'w1']:
    total_rms_squared += (dn_dict[band] * 0.1*zp_err_dict[band])**2
total_rms = np.sqrt(total_rms_squared)
print('Total RMS: {:.2f}%'.format(100*total_rms))

Total RMS: 0.90%


In [44]:
np.sqrt((0.11*0.3)**2 + (1.4*0.3)**2 + (1.23*0.6)**2 + (2.89*0.1)**2)

0.897582308203543